In [1]:
API_KEY = "cf64399ff0e7ee8a2270a76b7e0767156b22c49ad79e6aa6a38836a80ee41aa1"

In [2]:
import requests

headers = {"X-API-Key": API_KEY}

def fetch_locations(country_ids: dict) -> dict:
    """Takes {country_name: country_id} and returns {country_name: list_of_locations}."""
    results = {}
    for country, country_id in country_ids.items():
        all_locations = []
        page = 1
        while True:
            response = requests.get(
                "https://api.openaq.org/v3/locations",
                params={"countries_id": country_id, "limit": 1000, "page": page,},# "providers_id": 119},
                headers=headers
            )
            data = response.json()
            page_results = data["results"]
            if not page_results:
                break
            all_locations.extend(page_results)
            print(f"{country} page {page}: fetched {len(page_results)} | total: {len(all_locations)}")
            page += 1
        results[country] = all_locations
    return results


In [3]:
codes={"Germany": 50, "France": 22, "Australia": 177, "Brazil": 45}


In [4]:
locations = fetch_locations(codes)

Germany page 1: fetched 744 | total: 744
France page 1: fetched 973 | total: 973
Australia page 1: fetched 314 | total: 314
Brazil page 1: fetched 115 | total: 115


In [5]:
locations

{'Germany': [{'id': 2669,
   'name': 'München/Stachus',
   'locality': 'München',
   'timezone': 'Europe/Berlin',
   'country': {'id': 50, 'code': 'DE', 'name': 'Germany'},
   'owner': {'id': 4, 'name': 'Unknown Governmental Organization'},
   'provider': {'id': 70, 'name': 'EEA'},
   'isMobile': False,
   'isMonitor': True,
   'instruments': [{'id': 2, 'name': 'Government Monitor'}],
   'sensors': [{'id': 7559,
     'name': 'co µg/m³',
     'parameter': {'id': 4,
      'name': 'co',
      'units': 'µg/m³',
      'displayName': 'CO mass'}},
    {'id': 4275101,
     'name': 'no µg/m³',
     'parameter': {'id': 19843,
      'name': 'no',
      'units': 'µg/m³',
      'displayName': 'NO mass'}},
    {'id': 6681,
     'name': 'no2 µg/m³',
     'parameter': {'id': 5,
      'name': 'no2',
      'units': 'µg/m³',
      'displayName': 'NO₂ mass'}},
    {'id': 5575,
     'name': 'o3 µg/m³',
     'parameter': {'id': 3,
      'name': 'o3',
      'units': 'µg/m³',
      'displayName': 'O₃ mass'}},

In [6]:
def filter_monitors(locations: dict) -> dict:
    return {country: [loc for loc in locs if loc["isMonitor"]] for country, locs in locations.items()}

monitor_locations = filter_monitors(locations)
{country: len(locs) for country, locs in monitor_locations.items()}


{'Germany': 523, 'France': 828, 'Australia': 238, 'Brazil': 101}

In [7]:
location_ids = {country: [loc["id"] for loc in locs] for country, locs in monitor_locations.items()}
location_ids

{'Germany': [2669,
  2709,
  2916,
  2917,
  2932,
  2933,
  2935,
  2936,
  2937,
  2938,
  2939,
  2940,
  2941,
  2948,
  2949,
  2962,
  2963,
  2964,
  2965,
  2967,
  2968,
  2969,
  2970,
  2971,
  2973,
  2993,
  3009,
  3010,
  3011,
  3012,
  3013,
  3014,
  3015,
  3016,
  3017,
  3018,
  3019,
  3020,
  3021,
  3022,
  3023,
  3024,
  3025,
  3026,
  3027,
  3028,
  3029,
  3030,
  3031,
  3032,
  3033,
  3034,
  3035,
  3036,
  3037,
  3038,
  3039,
  3040,
  3041,
  3042,
  3044,
  3045,
  3046,
  3047,
  3048,
  3049,
  3050,
  3051,
  3052,
  3053,
  3054,
  3055,
  3056,
  3057,
  3058,
  3059,
  3060,
  3061,
  3062,
  3063,
  3064,
  3065,
  3066,
  3067,
  3068,
  3069,
  3070,
  3071,
  3072,
  3073,
  3074,
  3075,
  3076,
  3077,
  3078,
  3079,
  3080,
  3081,
  3082,
  3083,
  3084,
  3085,
  3086,
  3087,
  3088,
  3089,
  3090,
  3091,
  3092,
  3093,
  3094,
  3095,
  3096,
  3097,
  3098,
  3099,
  3100,
  3101,
  3102,
  3103,
  3104,
  3105,
  3106,
  310

In [8]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import os
from tqdm import tqdm

BUCKET = "openaq-data-archive"
YEARS = range(2022, 2025)
OUT_DIR = "./openaq_raw2"
os.makedirs(OUT_DIR, exist_ok=True)
# One client shared across threads (boto3 clients are thread-safe)
s3 = boto3.client("s3", config=Config(
    signature_version=UNSIGNED,
    max_pool_connections=128,
))

def download_location_year(args):
    country, loc_id, year = args
    prefix = f"records/csv.gz/locationid={loc_id}/year={year}/"
    paginator = s3.get_paginator("list_objects_v2")

    downloaded = 0
    for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            local_path = os.path.join(OUT_DIR, country, key)
            if os.path.exists(local_path):
                continue
            os.makedirs(os.path.dirname(local_path), exist_ok=True)
            s3.download_file(BUCKET, key, local_path)
            downloaded += 1
    return country, loc_id, year, downloaded

tasks = [
    (country, loc_id, year)
    for country, ids in location_ids.items()
    for loc_id in ids
    for year in YEARS
]

with ThreadPoolExecutor(max_workers=128) as executor:
    futures = {executor.submit(download_location_year, t): t for t in tasks}
    with tqdm(total=len(tasks), desc="Downloading", unit="task") as pbar:
        for future in as_completed(futures):
            country, loc_id, year, n = future.result()
            pbar.update(1)

Downloading: 100%|██████████| 5070/5070 [1:09:25<00:00,  1.22task/s]  
